# m38 lab · Pulses, windows and alerts

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You time pulses against heartbeats, test the "k windows in a row" rule that wearables use, simulate the review
inbox of a remote-monitoring clinic, and then run the same window detector on real Holter beat annotations.

| Part | What you do | Data |
| --- | --- | --- |
| A | Build ECG and PPG timing; check eq-m38-ibi and eq-m38-ibi-var | synthetic |
| B | A windowed irregularity detector on a day of rhythm: estimate q and p | synthetic |
| C | The k-in-a-row rule: check eq-m38-runs and eq-m38-detect; see what correlated noise does | synthetic |
| D | A clinic inbox with a triage score: review hours and hidden true alerts | synthetic |
| E | The window detector on the MIT-BIH Atrial Fibrillation Database | PhysioNet |
| F | Optional: a small network on RR-interval windows, in PyTorch | synthetic |

The synthetic rhythm and PPG are approximate models. Heart rates, interval variability, pulse delays, motion
bursts, alert rates and triage scores are illustrative values, not physiological or clinical references.

Each part ends with **checks**: lines that pass with the code as given. If you change something and a check
fails, that is information, not an error to hide. The lab runs in about a minute.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import ml
from epml import wearables as wr
from epml.data import DataUnavailable

## Part A · Pulse timing is not heartbeat timing

Take an hour of regular rhythm. Each pulse reaches the wrist a delay after its R peak: here a mean of 0.25 s with
a beat-to-beat SD of 20 ms and lag-one correlation 0.3 (illustrative). Check the identity
$\mathrm{IBI}_i = RR_i + \Delta_{i+1} - \Delta_i$ (eq-m38-ibi) and the variance formula (eq-m38-ibi-var).

In [ ]:
rng = np.random.default_rng(38)
beats, _ = wr.rhythm_day(hours=1.0, hr_bpm=70, rng=rng)
rr = np.diff(beats)
delays = wr.ar1_delays(beats.size, 0.25, 0.020, 0.3, rng)
pulses = beats + delays
ibi = np.diff(pulses)

predicted = wr.ppg_interval_variance(rr.var(), delays.var(), 0.3)
print(f"heart rate from RR {wr.heart_rate(rr):.1f} bpm, from pulses {wr.heart_rate(ibi):.1f} bpm")
print(f"CV of RR {rr.std() / rr.mean():.3f}, CV of pulse intervals {ibi.std() / ibi.mean():.3f}")
print(f"Var(IBI) {ibi.var():.2e} s^2, eq-m38-ibi-var predicts {predicted:.2e} s^2")

x, onsets = wr.synth_ppg(beats[:12], delays[:12], fs=50, noise=0.02, rng=rng)
t = np.arange(x.size) / 50
fig, ax = plt.subplots()
ax.plot(t, x, label="synthetic PPG")
ax.vlines(beats[:12], -0.2, 1.2, colors="gray", linestyles=":", label="R peaks")
ax.set(xlabel="time (s)", ylabel="PPG (arbitrary units)", xlim=(beats[0], beats[11]), title="Each pulse lags its beat")
ax.legend()
plt.show()

In [ ]:
# Checks
assert np.allclose(ibi, wr.ppg_intervals(rr, delays))
assert abs(ibi.var() / predicted - 1) < 0.15                 # one hour is a finite sample
assert ibi.std() / ibi.mean() > rr.std() / rr.mean()          # pulses look more irregular than beats
assert abs(wr.heart_rate(ibi) - wr.heart_rate(rr)) < 0.5      # but the mean rate is the same

## Part B · A window detector

Simulate a day: regular rhythm with two AF episodes. Pulse times carry delay jitter, and some 5-minute windows hold
a motion burst that adds extra timing noise. Bursts last one to four windows. For each window, the detector
computes the coefficient of variation (CV) of the pulse intervals and flags it if the CV exceeds a threshold.

In [ ]:
WIN = 300.0                         # seconds per window
def simulate_day(rng, af=((9.0, 2.0), (18.0, 0.5)), burst_rate=0.02, burst_sd=0.06):
    beats, in_af = wr.rhythm_day(hours=24, hr_bpm=rng.uniform(60, 80), af_episodes=af, rng=rng)
    d = wr.ar1_delays(beats.size, 0.25, 0.020, 0.3, rng)
    n = int(24 * 3600 // WIN)
    # motion bursts: a burst starts in a window with probability burst_rate and lasts 1-4 windows
    noisy = np.zeros(n, bool)
    for w in np.flatnonzero(rng.random(n) < burst_rate):
        noisy[w: w + rng.integers(1, 5)] = True
    w_of_beat = np.minimum(((beats - beats[0]) // WIN).astype(int), n - 1)
    d = d + np.where(noisy[w_of_beat], rng.normal(0, burst_sd, beats.size), 0.0)
    pulses = np.sort(beats + d)
    cv = wr.window_cv(pulses, WIN)
    starts = pulses[0] + WIN * np.arange(cv.size)
    af_win = np.array([any(a * 3600 <= s and s + WIN <= (a + l) * 3600 for a, l in af) for s in starts])
    return cv, af_win, noisy[: cv.size]

rng = np.random.default_rng(380)
days = [simulate_day(rng) for _ in range(6)]
cv = np.concatenate([d[0] for d in days])
af_win = np.concatenate([d[1] for d in days])
ok = ~np.isnan(cv)
auc = ml.auc_rank(cv[ok], af_win[ok].astype(int))
THRESH = np.quantile(cv[ok & ~af_win], 0.95)      # flags 5% of regular windows on these days
q = float(np.mean(cv[ok & ~af_win] > THRESH))
p = float(np.mean(cv[ok & af_win] > THRESH))
print(f"window AUC {auc:.3f}; threshold CV {THRESH:.3f}: q = {q:.3f} (regular), p = {p:.3f} (AF)")

fig, ax = plt.subplots()
ax.hist(cv[ok & ~af_win], bins=60, alpha=0.6, label="regular windows", density=True)
ax.hist(cv[ok & af_win], bins=30, alpha=0.6, label="AF windows", density=True)
ax.axvline(THRESH, color="k", ls="--", label="threshold")
ax.set(xlabel="CV of pulse intervals in a 5-min window", ylabel="density")
ax.legend()
plt.show()

In [ ]:
# Checks
assert auc > 0.95
assert 0.03 < q < 0.07 and p > 0.9

## Part C · k windows in a row

First check the formulas with independent windows: simulate many days of 288 windows, each false with probability
q from Part B. The synthetic AF of Part B was easy (p = 1), so for episodes take a detector that flags only 70% of AF
windows (p = 0.7, an illustrative value; Part E measures p on real data), and episodes of 6 windows (30 minutes).
Then count false notifications on the simulated days of Part B, where motion bursts make false windows come in
clumps.

In [ ]:
n = 288
ks = [1, 2, 3, 4, 6]
sim_rng = np.random.default_rng(3800)
iid = {k: np.mean([wr.runs_at_least(sim_rng.random(n) < q, k) for _ in range(3000)]) for k in ks}
formula = {k: wr.expected_runs(n, q, k) for k in ks}
L, P_AF = 6, 0.7
caught = {k: np.mean([wr.runs_at_least(sim_rng.random(L) < P_AF, k) > 0 for _ in range(4000)]) for k in ks}

# the simulated days: count runs within regular stretches only
clumped = {}
for k in ks:
    total = 0
    for cvd, afd, _ in days:
        flags = np.nan_to_num(cvd, nan=0.0) > THRESH
        flags[afd] = False
        total += wr.runs_at_least(flags, k)
    clumped[k] = total / len(days)
print(" k | formula/day | independent sim | clumped days | P(30-min episode caught) formula, sim")
for k in ks:
    print(f"{k:2d} | {formula[k]:11.4f} | {iid[k]:15.4f} | {clumped[k]:12.3f} | "
          f"{wr.detection_probability(L, P_AF, k):.3f}, {caught[k]:.3f}")

In [ ]:
# Checks
assert abs(iid[1] / formula[1] - 1) < 0.05 and abs(iid[2] / formula[2] - 1) < 0.15
assert all(abs(caught[k] - wr.detection_probability(L, P_AF, k)) < 0.03 for k in ks)
assert wr.detection_probability(L, P_AF, 6) < 0.2 < wr.detection_probability(L, P_AF, 2)   # large k misses episodes
assert clumped[3] > 3 * formula[3]      # clumped noise beats the independence formula

With independent windows, two or three windows in a row nearly remove false notifications. The simulated days
disagree: motion bursts last several windows, so false windows arrive together and survive the rule far more
often. The formula is a lower bound when noise is clumped.

## Part D · The clinic inbox

A clinic follows 500 implanted monitors for two months. Each patient sends 0.1 alerts a day; 25% are true (the
historical figure of about 75% false alerts, Neiman et al. 2024). A triage model gives each alert a score; true
alerts score higher on average (an illustrative, made-up separation). Pick the threshold on month 1 so that at
least 98% of true alerts are kept, then apply it to month 2.

In [ ]:
rng = np.random.default_rng(3801)
N, RATE, PPV, MIN = 500, 0.1, 0.25, 3.0

def month(rng, days=30):
    n_alerts = rng.poisson(N * RATE * days)
    true = rng.random(n_alerts) < PPV
    score = rng.normal(np.where(true, 2.0, 0.0), 1.0)
    return true, score

true1, s1 = month(rng)
true2, s2 = month(rng)
thr = np.quantile(s1[true1], 0.02)               # keep 98% of month-1 true alerts
kept_true = np.mean(s2[true2] >= thr)
removed_false = np.mean(s2[~true2] < thr)
ppv_new, shown, lost = wr.filtered_alerts(PPV, kept_true, removed_false)
load = wr.alert_workload(N, RATE, PPV, MIN)
observed_ppv = np.mean(true2[s2 >= thr])
hidden = int(np.sum(true2 & (s2 < thr)))
print(f"without triage: {load['alerts']:.0f} alerts/day, {load['hours']:.1f} h/day of review")
print(f"month 2 with triage: keeps {kept_true:.1%} of true alerts, removes {removed_false:.1%} of false alerts")
print(f"PPV {observed_ppv:.2f} observed, {ppv_new:.2f} from eq-m38-filter; review {load['hours'] * shown:.1f} h/day; "
      f"{hidden} true alerts hidden in the month")

In [ ]:
# Checks
assert abs(observed_ppv - ppv_new) < 0.03
assert kept_true > 0.95 and removed_false > 0.3
assert hidden > 0          # a filter that saves hours also hides true alerts

## Part E · Real Holter recordings

The MIT-BIH Atrial Fibrillation Database holds long ambulatory ECG recordings of people with AF, mostly
paroxysmal (PhysioNet, doi:10.13026/C2MW2D; Open Data Commons Attribution v1.0; Moody and Mark 1983). We use only
its beat annotations (.qrs, unaudited) and manual rhythm annotations (.atr), a few hundred kilobytes for five
records, verified against PhysioNet's published SHA-256 list. Without internet access this part is skipped.

The detector is the one from Part B, now on beat times rather than pulse times. The threshold is set on the
first two records and tested on the other three.

In [ ]:
from epml.data import load_afdb_rr

try:
    afdb = load_afdb_rr()
    REAL = True
except DataUnavailable as err:
    print("Part E skipped:", err)
    REAL = False

def windows(rec):
    beats = rec["beats"]
    cv = wr.window_cv(beats, WIN)
    starts = beats[0] + WIN * np.arange(cv.size)
    af = np.array([sum(max(0.0, min(b, s + WIN) - max(a, s)) for a, b in rec["af"]) > WIN / 2 for s in starts])
    return cv, af

if REAL:
    names = sorted(afdb)
    per = {r: windows(afdb[r]) for r in names}
    for r in names:
        cvr, afr = per[r]
        print(f"record {r}: {cvr.size} windows ({cvr.size * WIN / 3600:.1f} h), {afr.mean():.0%} in AF")
    tune = names[:2]
    test = names[2:]
    cv_t = np.concatenate([per[r][0] for r in tune]); af_t = np.concatenate([per[r][1] for r in tune])
    ok_t = ~np.isnan(cv_t)
    thr_real = np.quantile(cv_t[ok_t & ~af_t], 0.95)
    cv_e = np.concatenate([per[r][0] for r in test]); af_e = np.concatenate([per[r][1] for r in test])
    ok_e = ~np.isnan(cv_e)
    auc_real = ml.auc_rank(cv_e[ok_e], af_e[ok_e].astype(int))
    q_real = float(np.mean(cv_e[ok_e & ~af_e] > thr_real))
    p_real = float(np.mean(cv_e[ok_e & af_e] > thr_real)) if (ok_e & af_e).any() else float("nan")
    # are false windows clumped? lag-one correlation of flags in consecutive regular windows
    pairs = []
    for r in test:
        cvr, afr = per[r]
        f = np.nan_to_num(cvr, nan=0.0) > thr_real
        for i in range(len(f) - 1):
            if not afr[i] and not afr[i + 1]:
                pairs.append((f[i], f[i + 1]))
    pairs = np.array(pairs, float)
    rho_flags = np.corrcoef(pairs[:, 0], pairs[:, 1])[0, 1] if pairs[:, 0].std() > 0 else 0.0
    print(f"test records: window AUC {auc_real:.3f}; q = {q_real:.3f}, p = {p_real:.3f}; "
          f"lag-one correlation of false flags {rho_flags:.2f}")

In [ ]:
# Checks
if REAL:
    assert len(afdb) == 5 and all(per[r][0].size > 50 for r in names)
    assert auc_real > 0.8
    assert 0 <= q_real <= 1

## Part F · Optional: a small network on interval windows, in PyTorch

A one-dimensional convolutional network reads 64 consecutive RR intervals and outputs a probability of AF. The
negatives include regular rhythm with premature beats, which fool a simple CV threshold. This part runs only if
PyTorch is installed (`pip install torch`).

In [ ]:
def interval_windows(rng, n_each=300, length=64):
    X, y = [], []
    for label, kind in [(0, "sinus"), (0, "ectopy"), (1, "af")]:
        for _ in range(n_each):
            mean = 60 / rng.uniform(55, 110)
            if kind == "af":
                k = 1 / 0.2**2
                rr = rng.gamma(k, mean / k, size=length)
            else:
                rr = mean * (1 + 0.03 * rng.normal(size=length))
                if kind == "ectopy":
                    for i in np.flatnonzero(rng.random(length - 1) < 0.15):
                        rr[i], rr[i + 1] = 0.65 * mean, 1.35 * mean
            X.append(rr / rr.mean() - 1)
            y.append(label)
    return np.array(X), np.array(y)

rng = np.random.default_rng(3802)
Xw, yw = interval_windows(rng)
Xw_test, yw_test = interval_windows(rng, n_each=150)
auc_cv = ml.auc_rank(Xw_test.std(axis=1), yw_test)
print(f"CV threshold AUC on held-out windows: {auc_cv:.3f}")

try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: the network is skipped.")

if torch is not None:
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Conv1d(1, 16, 5), torch.nn.ReLU(), torch.nn.Conv1d(16, 16, 5), torch.nn.ReLU(),
                              torch.nn.AdaptiveAvgPool1d(1), torch.nn.Flatten(), torch.nn.Linear(16, 1))
    Xt = torch.tensor(Xw[:, None, :], dtype=torch.float32)
    yt = torch.tensor(yw, dtype=torch.float32)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    lossf = torch.nn.BCEWithLogitsLoss()
    for epoch in range(200):
        opt.zero_grad()
        loss = lossf(net(Xt).squeeze(1), yt)
        loss.backward()
        opt.step()
    with torch.no_grad():
        p_net = torch.sigmoid(net(torch.tensor(Xw_test[:, None, :], dtype=torch.float32)).squeeze(1)).numpy()
    auc_net = ml.auc_rank(p_net, yw_test)
    print(f"PyTorch network AUC on held-out windows: {auc_net:.3f}")

In [ ]:
# Checks
assert auc_cv < 0.95           # premature beats fool the CV threshold
if torch is not None:
    assert auc_net > auc_cv     # the network learns the early-then-late pattern

## Where to go next

- In Part B, make motion bursts longer and rerun Part C: how does the gap between formula and simulation grow?
- In Part D, demand 99.5% of true alerts kept. What happens to review hours?
- In Part E, change the window to 2 minutes. Does the window AUC change? Does the flag correlation?